# 04 · New ideas

Four ideas that the earlier rounds never tested, each aimed at information the base models cannot see.
A–C were tested first; D came out of a closer look at B and was tested in a second round (end of this notebook).

| Idea | Uses labels? | Where it is computed |
| --- | --- | --- |
| **A. Population statistics** over train + test (frequencies, group-relative amounts, value popularity, repeat counts) | no | notebook 02 (`plus` columns) |
| **B. Repeat-applicant history**: default labels of *other* applications by the same probable person | yes | inside each CV fold |
| **C. Neighbour default rate**: default rate of the most similar applicants in external scores and loan term | yes | inside each CV fold |
| **D. Same-person history (round 2)**: B with a stricter same-person match, earlier/later and duplicate applications | yes | inside each CV fold |

**Leakage control for B, C and D.** Inside a fold, only labels of the fold's training part are used.
A training row never sees its own label: its features come from the *other* labelled rows
(its group-mates, its neighbours excluding itself). Validation and test rows use all training-part rows.
So every row gets "labels of other applicants", the same thing the test rows get.

**Decision rule (set before running).** Ideas are screened with LightGBM (fast) on two seeds against a control
using the same folds and settings. An idea is kept if it improves OOF AUC by at least **0.0002**. Kept ideas are
then confirmed on CatBoost, and only confirmed improvements go to the final blend (notebook 05).

In [1]:
import sys
sys.path.insert(0, '..')
from functools import lru_cache
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from src.utils import load_features, auc, WORK
from src.cv import run_cv, folds, median_iterations
from src import models

train, test, sets = load_features()
y = train.TARGET.to_numpy().astype(int)
BASE, RECON, PLUS, PAIR = sets['base'], sets['base'] + sets['recon'], sets['plus'], sets['pair']
LGB_PARAMS = dict(num_leaves=21, min_child_samples=89, colsample_bytree=0.685, subsample=0.962, subsample_freq=1,
                  reg_lambda=2.13, reg_alpha=3.69)
CAT_PARAMS = dict(depth=5, l2_leaf_reg=8, random_strength=0.5)
SCREEN_SEEDS = (0, 1)

## Label-based features (B and C)
**B.** For each row: number of other labelled applications with the same `APPLICANT_KEY`, how many of them
defaulted, and the rate.

**C.** Similarity space: the three external scores (missing values replaced by their reconstruction from
notebook 02) and the loan term, each standardised. For each row, the default rate among its 100 and 500
nearest training-part applicants.

In [2]:
def repeat_features(keys_lab, y_lab, keys_query, query_is_labelled):
    stats = pd.DataFrame({'key': keys_lab, 'y': y_lab}).groupby('key').y.agg(['size', 'sum'])
    n = keys_query.map(stats['size']).fillna(0).to_numpy()
    s = keys_query.map(stats['sum']).fillna(0).to_numpy()
    if query_is_labelled is not None:              # remove the row's own label
        n, s = n - 1, s - query_is_labelled
    return pd.DataFrame({'REPEAT_LABELLED': n, 'REPEAT_DEFAULTS': s, 'REPEAT_RATE': np.where(n > 0, s / np.maximum(n, 1), np.nan)})


EDGES = pd.read_parquet(WORK / 'pairs.parquet')       # same-person pairs from notebook 02 (row numbers: train, then test)
Y_ALL = np.r_[y, np.full(len(test), np.nan)]


def person_features(labelled_rows):
    """Labels of the *other* applications of the same person that are in `labelled_rows` (D).

    Pairs never link a row to itself, so a training row cannot see its own label.
    Missing (NaN) means the row has no such labelled application."""
    labelled = np.zeros(len(Y_ALL), bool)
    labelled[labelled_rows] = True
    e = EDGES[labelled[EDGES.m.to_numpy()]].assign(ym=lambda d: Y_ALL[d.m.to_numpy()])
    F = pd.DataFrame(index=range(len(Y_ALL)))
    g = e.groupby('r').ym
    F['PERSON_LABELLED'], F['PERSON_DEFAULTS'] = g.size(), g.sum()
    F[['PERSON_LABELLED', 'PERSON_DEFAULTS']] = F[['PERSON_LABELLED', 'PERSON_DEFAULTS']].fillna(0)
    F['PERSON_RATE'] = np.where(F.PERSON_LABELLED > 0, F.PERSON_DEFAULTS / F.PERSON_LABELLED.clip(lower=1), np.nan)
    for name, part in [('LATER', e[e.m_later]), ('EARLIER', e[e.m_earlier])]:
        F[f'PERSON_DEFAULTS_{name}'] = part.groupby('r').ym.sum()
        F[f'PERSON_LABELLED_{name}'] = part.groupby('r').ym.size()
    F['PERSON_DEFAULTS_DUPLICATE'] = e[e.duplicate].groupby('r').ym.sum()
    F['PERSON_DEFAULTS_ALL_4_MATCH'] = e[e.n_match >= 4].groupby('r').ym.sum()
    F['PERSON_GAP_TO_DEFAULT'] = e[e.ym == 1].groupby('r').gap.min()
    return F


def knn_space(df):
    cols = []
    for c in ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']:
        cols.append(df[c].fillna(df['RECON_' + c]))
    cols.append(df.CREDIT_TERM.fillna(df.CREDIT_TERM.median()))
    return np.column_stack(cols)


SPACE_ALL = knn_space(pd.concat([train, test]))
SCALE = SPACE_ALL.std(axis=0)
SPACE_TRAIN, SPACE_TEST = knn_space(train) / SCALE, knn_space(test) / SCALE
KS = (100, 500)


def knn_features(index_points, y_index, query, exclude_self):
    k = max(KS) + int(exclude_self)
    finder = NearestNeighbors(n_neighbors=k, algorithm='kd_tree', n_jobs=8).fit(index_points)
    out = np.zeros((len(query), len(KS)))
    for start in range(0, len(query), 20000):
        ind = finder.kneighbors(query[start:start + 20000], return_distance=False)
        if exclude_self:                           # drop the row itself (query rows = index rows here)
            own = np.arange(start, start + len(ind))[:, None]
            keep = ind != own
            keep[~(~keep).any(axis=1), -1] = False  # self not among neighbours (exact duplicates): drop the farthest
            ind = ind[keep].reshape(len(ind), max(KS))
        labels = y_index[ind]
        for j, kk in enumerate(KS):
            out[start:start + len(ind), j] = labels[:, :kk].mean(axis=1)
    return pd.DataFrame(out, columns=[f'NEIGHBOUR_RATE_{kk}' for kk in KS])


@lru_cache(maxsize=None)
def label_features(seed, fold):
    tr, va = folds(y, seed)[fold]
    keys = train.APPLICANT_KEY
    parts = []
    for rows, query_keys, labelled, space, self_ in [(tr, keys.iloc[tr], y[tr], SPACE_TRAIN[tr], True),
                                                     (va, keys.iloc[va], None, SPACE_TRAIN[va], False),
                                                     (None, test.APPLICANT_KEY, None, SPACE_TEST, False)]:
        r = repeat_features(keys.iloc[tr], y[tr], query_keys, labelled)
        k = knn_features(SPACE_TRAIN[tr], y[tr], space, self_)
        parts.append(pd.concat([r, k], axis=1))
    person = person_features(tr)
    for i, rows in enumerate([tr, va, np.arange(len(train), len(Y_ALL))]):
        parts[i] = pd.concat([parts[i], person.iloc[rows].reset_index(drop=True)], axis=1)
    return parts


def augmented(columns, extra):
    """CV augment hook: base columns + selected fold-dependent label features."""
    def augment(tr, va, seed, fold):
        f_tr, f_va, f_te = label_features(seed, fold)
        join = lambda X, F: pd.concat([X[columns].reset_index(drop=True), F[extra]], axis=1)
        return join(train.iloc[tr], f_tr), join(train.iloc[va], f_va), join(test, f_te)
    return augment


REPEAT = ['REPEAT_LABELLED', 'REPEAT_DEFAULTS', 'REPEAT_RATE']
NEIGHBOUR = [f'NEIGHBOUR_RATE_{k}' for k in KS]
PERSON = list(person_features(np.arange(len(train))).columns)

Quick look before modelling: do the new label features separate defaulters on their own (seed 0, fold 0)?

In [3]:
tr, va = folds(y, 0)[0]
f_tr, f_va, _ = label_features(0, 0)
has_history = f_va.REPEAT_LABELLED.to_numpy() > 0
print(f'validation rows with a labelled repeat application: {has_history.mean():.2%}')
print('default rate | a previous application defaulted:',
      round(y[va][has_history & (f_va.REPEAT_DEFAULTS.to_numpy() > 0)].mean(), 3),
      '| none defaulted:', round(y[va][has_history & (f_va.REPEAT_DEFAULTS.to_numpy() == 0)].mean(), 3),
      '| overall:', round(y.mean(), 3))
for c in NEIGHBOUR:
    print(c, 'AUC on its own:', round(auc(y[va], f_va[c]), 4))
print('EXT_MEAN AUC on its own:', round(auc(y[va], -train.EXT_MEAN.iloc[va].fillna(0.5)), 4))

validation rows with a labelled repeat application: 2.69%
default rate | a previous application defaulted: 0.254 | none defaulted: 0.052 | overall: 0.081
NEIGHBOUR_RATE_100 AUC on its own: 0.7158
NEIGHBOUR_RATE_500 AUC on its own: 0.7243
EXT_MEAN AUC on its own: 0.7031


## Screening with LightGBM (development runs, seeds 0 and 1)

In [4]:
def lgb_run(name, columns, extra=()):
    extra = list(extra)
    _, _, info = run_cv(name, models.lightgbm(LGB_PARAMS), train[columns], y, test[columns], seeds=SCREEN_SEEDS,
                        augment=augmented(columns, extra) if extra else None, config=dict(params=LGB_PARAMS, extra=extra))
    return info


screen = {
    'control (base)': lgb_run('lgb_base_dev', BASE),
    'A  population statistics': lgb_run('lgb_plus_dev', BASE + PLUS),
    'B  repeat-applicant history': lgb_run('lgb_repeat_dev', BASE, REPEAT),
    'C  neighbour default rate': lgb_run('lgb_neighbour_dev', BASE, NEIGHBOUR),
    'A+B+C': lgb_run('lgb_abc_dev', BASE + PLUS, REPEAT + NEIGHBOUR),
}
control = screen['control (base)']['auc']
table = pd.DataFrame({name: dict(auc=i['auc'], gain=i['auc'] - control, seed0=i['seed_auc'][0], seed1=i['seed_auc'][1])
                      for name, i in screen.items()}).T
table.round(5)

lgb_base_dev: OOF AUC 0.76288  per seed [0.7615, 0.76176]  (1.5 min of fitting)
lgb_plus_dev: OOF AUC 0.76272  per seed [0.7615, 0.76148]  (1.6 min of fitting)
lgb_repeat_dev: OOF AUC 0.76423  per seed [0.76261, 0.763]  (3.4 min of fitting)
lgb_neighbour_dev: OOF AUC 0.76294  per seed [0.76164, 0.7613]  (1.6 min of fitting)


lgb_abc_dev: OOF AUC 0.76400  per seed [0.76252, 0.76222]  (1.8 min of fitting)


,auc,gain,seed0,seed1
control (base),0.76288,0.00000,0.76150,0.76176
A population statistics,0.76272,-0.00016,0.76150,0.76148
B repeat-applicant history,0.76423,0.00135,0.76261,0.76300
C neighbour default rate,0.76294,0.00006,0.76164,0.76130
A+B+C,0.76400,0.00112,0.76252,0.76222


Per-fold gains show whether an improvement is consistent or driven by one fold.

In [5]:
def fold_gain(info):
    a = pd.DataFrame(info['folds']).set_index(['seed', 'fold']).auc
    b = pd.DataFrame(screen['control (base)']['folds']).set_index(['seed', 'fold']).auc
    return (a - b)

pd.DataFrame({name: fold_gain(i) for name, i in screen.items() if name != 'control (base)'}).round(5)

A  population statistics  B  repeat-applicant history  \
seed fold                                                          
0    0                     -0.00012                      0.00009   
     1                      0.00012                      0.00098   
     2                      0.00054                      0.00070   
     3                     -0.00074                      0.00113   
     4                     -0.00001                      0.00265   
1    0                     -0.00107                      0.00205   
     1                      0.00042                      0.00135   
     2                     -0.00042                      0.00153   
     3                     -0.00013                      0.00172   
     4                     -0.00033                     -0.00044   

           C  neighbour default rate    A+B+C  
seed fold                                      
0    0                       0.00021  0.00150  
     1                       0.00154  0.00137  
     2                      -0.00094  0.00046  
     3                      -0.00068 -0.00085  
     4                       0.00054  0.00264  
1    0                      -0.00055  0.00051  
     1                      -0.00118 -0.00013  
     2                      -0.00133 -0.00003  
     3                       0.00029  0.00084  
     4                       0.00045  0.00094

## Decision and confirmation on CatBoost
Ideas passing the 0.0002 rule are combined; the combination is then run with CatBoost-recon
(the strongest family) against the CatBoost-recon development run from notebook 03.

In [6]:
kept = [name for name in ['A  population statistics', 'B  repeat-applicant history', 'C  neighbour default rate']
        if table.loc[name, 'gain'] >= 0.0002]
print('kept ideas:', kept)
KEEP_COLUMNS = PLUS if 'A  population statistics' in kept else []
KEEP_EXTRA = (REPEAT if 'B  repeat-applicant history' in kept else []) + (NEIGHBOUR if 'C  neighbour default rate' in kept else [])

kept ideas: ['B  repeat-applicant history']


In [7]:
if kept:
    lgb_kept = lgb_run('lgb_kept_dev', BASE + KEEP_COLUMNS, KEEP_EXTRA)
    print(f'LightGBM kept-ideas gain: {lgb_kept["auc"] - control:+.5f}')
    cols = RECON + KEEP_COLUMNS
    _, _, cat_new_dev = run_cv('cat_new_dev', models.catboost(CAT_PARAMS), train[cols], y, test[cols],
                               augment=augmented(cols, KEEP_EXTRA) if KEEP_EXTRA else None,
                               config=dict(params=CAT_PARAMS, extra=KEEP_EXTRA))
    _, _, cat_ref = run_cv('cat_recon_dev', models.catboost(CAT_PARAMS), train[RECON], y, test[RECON], config=dict(params=CAT_PARAMS))
    print(f'CatBoost-recon gain from kept ideas (seed 0): {cat_new_dev["auc"] - cat_ref["auc"]:+.5f}')

lgb_kept_dev: OOF AUC 0.76423  per seed [0.76261, 0.763]  (1.6 min of fitting)
LightGBM kept-ideas gain: +0.00135
cat_new_dev: OOF AUC 0.76458  per seed [0.76458]  (10.0 min of fitting)
cat_recon_dev: OOF AUC 0.76346  per seed [0.76346]  (10.4 min of fitting)
CatBoost-recon gain from kept ideas (seed 0): +0.00112


## Final runs of the improved families (fixed length, 3 seeds)
Same protocol as notebook 03. Whether these replace or join the base models is decided in notebook 05
by the blend.

In [8]:
SEEDS = (0, 1, 2)
if kept:
    n = median_iterations(lgb_kept)
    run_cv('lgb_new', models.lightgbm(LGB_PARAMS, iterations=n), train[BASE + KEEP_COLUMNS], y, test[BASE + KEEP_COLUMNS],
           seeds=SEEDS, augment=augmented(BASE + KEEP_COLUMNS, KEEP_EXTRA) if KEEP_EXTRA else None,
           config=dict(params=LGB_PARAMS, iterations=n, extra=KEEP_EXTRA))
    n = median_iterations(cat_new_dev)
    run_cv('cat_new', models.catboost(CAT_PARAMS, iterations=n), train[cols], y, test[cols], seeds=SEEDS,
           augment=augmented(cols, KEEP_EXTRA) if KEEP_EXTRA else None,
           config=dict(params=CAT_PARAMS, iterations=n, extra=KEEP_EXTRA))

lgb_new: OOF AUC 0.76438  per seed [0.76238, 0.76267, 0.76268]  (2.8 min of fitting)


cat_new: OOF AUC 0.76630  per seed [0.76427, 0.76512, 0.76458]  (24.3 min of fitting)


The improved tree models fully replaced the base tree models in a preliminary blend, so the neural network
gets the same kept features (same 40-epoch recipe as notebook 03).

In [9]:
if KEEP_EXTRA:
    run_cv('tabm_new', models.tabm(iterations=40), train[BASE + KEEP_COLUMNS], y, test[BASE + KEEP_COLUMNS], seeds=SEEDS,
           augment=augmented(BASE + KEEP_COLUMNS, KEEP_EXTRA), config=dict(epochs=40, extra=KEEP_EXTRA))

tabm_new: OOF AUC 0.76524  per seed [0.76204, 0.761, 0.76111]  (43.5 min of fitting)


## Round 2 · D. Same-person history with a stricter match

A closer look at the `APPLICANT_KEY` groups (notebook 02, section 5) showed that many of them join different people.
Notebook 02 now builds stricter **same-person pairs** (3 of 4 constant date differences, or 2 plus the same region
and education) and 5 label-free `pair` columns. Here the pairs' labels become fold-dependent features, with the same
leakage rules as B:

* number of labelled matched applications, their defaults and default rate;
* defaults among **later** and **earlier** applications separately (later applications default more often:
  7.9% vs 5.5% in matched labelled pairs);
* defaults among exact duplicates (same birth date) and among pairs matching on all 4 differences;
* gap in days to the nearest defaulted application.

D is added **on top of** B (the kept idea), and the bar is the same: at least +0.0002 over B (LightGBM, seeds 0 and 1).

In [10]:
d_dev = lgb_run('lgb_person_dev', BASE + PAIR, REPEAT + PERSON)
d_only_pairs = lgb_run('lgb_pair_cols_dev', BASE + PAIR, REPEAT)      # label-free pair columns alone, for reference
b_dev = screen['B  repeat-applicant history']
round2 = pd.DataFrame({name: dict(auc=i['auc'], gain_vs_B=i['auc'] - b_dev['auc'], gain_vs_control=i['auc'] - control,
                                  seed0=i['seed_auc'][0], seed1=i['seed_auc'][1])
                       for name, i in {'B (kept)': b_dev, 'B + pair columns': d_only_pairs, 'B + D (pair columns + same-person labels)': d_dev}.items()}).T
round2.round(5)

lgb_person_dev: OOF AUC 0.76620  per seed [0.76471, 0.76471]  (1.6 min of fitting)
lgb_pair_cols_dev: OOF AUC 0.76610  per seed [0.76489, 0.76435]  (1.6 min of fitting)


,auc,gain_vs_B,gain_vs_control,seed0,seed1
B (kept),0.76423,0.00000,0.00135,0.76261,0.76300
B + pair columns,0.76610,0.00187,0.00322,0.76489,0.76435
B + D (pair columns + same-person labels),0.76620,0.00197,0.00332,0.76471,0.76471


In [11]:
pd.DataFrame({'B + D gain over B': fold_gain(d_dev) - fold_gain(b_dev)}).round(5).T

seed                     0                                           1  \
fold                     0        1       2        3        4        0   
B + D gain over B  0.00258  0.00132  0.0019  0.00244  0.00225  0.00022   

seed                                                   
fold                     1        2        3        4  
B + D gain over B  0.00209  0.00105  0.00126  0.00372

**Reading the round-2 results.**

* D improves on B by **+0.0020**, about as much as all of round 1 together, and it does so on **all 10 folds**
  (+0.0002 to +0.0037). It is a consistent effect, not one lucky fold.
* Most of the gain comes from the **label-free pair columns** (+0.0019). The pair labels add little on top (+0.0001).
  The pair structure itself is informative, without any labels (training rows): an application that follows an
  earlier one by the same person defaults 9.8% of the time, one that was followed by a later application only 3.5%
  (base rate 8.1%), and reapplying within about two months goes with 10.4%. That also makes the gain less likely to
  be an artefact of the training labels.
* D passes the decision rule and is confirmed on CatBoost next.

In [12]:
keep_d = d_dev['auc'] - b_dev['auc'] >= 0.0002
print('keep D:', keep_d)
V2_EXTRA = REPEAT + PERSON
cols_v2 = RECON + PAIR
if keep_d:
    _, _, cat_v2_dev = run_cv('cat_v2_dev', models.catboost(CAT_PARAMS), train[cols_v2], y, test[cols_v2],
                              augment=augmented(cols_v2, V2_EXTRA), config=dict(params=CAT_PARAMS, extra=V2_EXTRA))
    print(f'CatBoost-recon gain from D over B (seed 0): {cat_v2_dev["auc"] - cat_new_dev["auc"]:+.5f}')

keep D: True
cat_v2_dev: OOF AUC 0.76643  per seed [0.76643]  (11.4 min of fitting)
CatBoost-recon gain from D over B (seed 0): +0.00185


**Confirmed.** CatBoost-recon gains as much as LightGBM (+0.0019 over B, seed 0). The final runs below use D.

### Final runs of the v2 families (fixed length; more seeds for CatBoost and TabM)
LightGBM uses 3 seeds, CatBoost 6 and TabM 9. Averaging more seeds (each with its own fold split) helped the two
models that carry most of the blend weight: with 3 seeds each the blend scored 0.77019 (nested), with 6 CatBoost
and 9 TabM seeds 0.77066. Five CatBoost settings (depth 6–7, stronger L2, 254 borders, bagging) were also tried on
the v2 features (seed 0, development runs); none beat the current one by more than 0.0001.

TabM uses the same 40-epoch recipe. On Apple silicon it can run on the GPU (`HC_DEVICE=mps`), which is about twice as
fast as the CPU; this changes the floating-point details, so results then match a CPU run only approximately.

In [13]:
if keep_d:
    n = median_iterations(d_dev)
    run_cv('lgb_v2', models.lightgbm(LGB_PARAMS, iterations=n), train[BASE + PAIR], y, test[BASE + PAIR], seeds=SEEDS,
           augment=augmented(BASE + PAIR, V2_EXTRA), config=dict(params=LGB_PARAMS, iterations=n, extra=V2_EXTRA))
    n = median_iterations(cat_v2_dev)
    run_cv('cat_v2', models.catboost(CAT_PARAMS, iterations=n), train[cols_v2], y, test[cols_v2], seeds=tuple(range(6)),
           augment=augmented(cols_v2, V2_EXTRA), config=dict(params=CAT_PARAMS, iterations=n, extra=V2_EXTRA))
    run_cv('tabm_v2', models.tabm(iterations=40), train[BASE + PAIR], y, test[BASE + PAIR], seeds=tuple(range(9)),
           augment=augmented(BASE + PAIR, V2_EXTRA), config=dict(epochs=40, extra=V2_EXTRA))

lgb_v2: OOF AUC 0.76634  per seed [0.76446, 0.76441, 0.76444]  (1.7 min of fitting)


cat_v2: OOF AUC 0.76889  per seed [0.7662, 0.7668, 0.76625, 0.76729, 0.76656, 0.76657]  (100.7 min of fitting)


tabm_v2: OOF AUC 0.76833  per seed [0.76356, 0.7627, 0.76279, 0.76284, 0.76318, 0.76289, 0.76098, 0.76138, 0.76532]  (263.8 min of fitting)
